# 📘 Módulo 05 - Notebook 01: Merge, Join y Conciliación Bancaria

## 🔗 Unión de DataFrames y Conciliaciones Automáticas

**Libro:** Saliendo de lo Pandito  
**Módulo:** 05 - Reshaping y Conciliaciones  
**Duración estimada:** 75 minutos  
**Dificultad:** 🟠 Intermedio-Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos

✅ **Dominar** los 4 tipos de joins (inner, left, right, outer)  
✅ **Aplicar** merge con múltiples claves  
✅ **Validar** integridad de merges (validate, indicator)  
✅ **Automatizar** conciliación bancaria  
✅ **Identificar** diferencias y partidas pendientes

---

## 📚 Contenido

1. Teoría de Merge y Join
2. Los 4 Tipos de Join
3. Merge en Acción
4. Múltiples Claves
5. Validación de Merges
6. Introducción a Conciliación Bancaria
7. Caso Integrador: Conciliación Automática

---

## 💡 Por Qué Importa

**Problema empresarial típico:**

* 📚 Libro Diario: 1,500 movimientos
* 🏬 Extracto Bancario: 1,450 movimientos
* ❓ **¿Dónde están las 50 diferencias?**

**Solución:** Merge + lógica de conciliación = **Conciliación automática en segundos**


In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("🔗 MERGE, JOIN Y CONCILIACIÓN BANCARIA")
print("="*70)
print("• Merge: Unir DataFrames por columnas comunes")
print("• Join: 4 tipos (inner, left, right, outer)")
print("• Conciliación: Detectar diferencias automáticamente")
print("\n📖 Métodos clave:")
print("  - pd.merge(df1, df2, on='col', how='inner')")
print("  - pd.merge(..., how='left|right|outer')")
print("  - pd.merge(..., indicator=True)")
print("  - pd.merge(..., validate='one_to_one')")
print("="*70)
print("✅ Librerías cargadas")



## 📚 Teoría: Merge vs Join vs Concat

### 🔗 Merge (Unión por Columnas Comunes)

**Concepto:** Unir 2 DataFrames usando **columna(s) clave(s)** compartidas

```python
pd.merge(df1, df2, on='columna_clave', how='inner')
```

**Análogo SQL:**
```sql
SELECT * FROM df1 INNER JOIN df2 ON df1.col = df2.col
```

---

### 📦 Concat (Unión por Posición)

**Concepto:** Apilar DataFrames **verticalmente** (filas) u **horizontalmente** (columnas)

```python
pd.concat([df1, df2], axis=0)  # vertical (filas)
pd.concat([df1, df2], axis=1)  # horizontal (columnas)
```

**Diferencia clave:**
* `merge`: Busca coincidencias en columnas clave
* `concat`: Solo apila (no busca coincidencias)

---

### 📊 Los 4 Tipos de Join

| Tipo | Descripción | Uso Típico |
|------|-------------|-------------|
| **inner** | Solo filas con coincidencia en AMBOS DataFrames | Facturas pagadas |
| **left** | TODAS las filas de df1 + coincidencias de df2 | Clientes con/sin pedidos |
| **right** | TODAS las filas de df2 + coincidencias de df1 | Pedidos con/sin factura |
| **outer** | TODAS las filas de AMBOS DataFrames | Conciliación bancaria |

---

### 💡 Regla de Oro

👉 **Siempre preguntarse: "¿Qué quiero conservar?"**

* ¿Solo coincidencias? → **inner**
* ¿Todo de la izquierda? → **left**
* ¿Todo de la derecha? → **right**
* ¿**Todo** (para ver diferencias)? → **outer**

In [0]:
import pandas as pd

print("🔗 LOS 4 TIPOS DE JOIN")
print("="*70)

# Datos de ejemplo: Facturas y Pagos
facturas = pd.DataFrame({
    'Factura': ['FAC-001', 'FAC-002', 'FAC-003'],
    'Cliente': ['Acme Corp', 'TechStart', 'FinPlus'],
    'Monto': [50000, 25000, 15000]
})

pagos = pd.DataFrame({
    'Factura': ['FAC-001', 'FAC-003', 'FAC-004'],
    'Fecha_Pago': ['2024-01-15', '2024-01-20', '2024-01-25'],
    'Monto_Pagado': [50000, 15000, 10000]
})

print("\n📄 Facturas:")
print(facturas)
print("\n📄 Pagos:")
print(pagos)

print("\n" + "="*70)
print("\n1️⃣  INNER JOIN (solo coincidencias)")
inner = pd.merge(facturas, pagos, on='Factura', how='inner')
print(inner)
print(f"\nResultado: {len(inner)} filas (FAC-001 y FAC-003 tienen pago)")

print("\n" + "-"*70)
print("\n2️⃣  LEFT JOIN (todas las facturas)")
left = pd.merge(facturas, pagos, on='Factura', how='left')
print(left)
print(f"\nResultado: {len(left)} filas (TODAS las facturas, con/sin pago)")
print("\u26a0️  FAC-002 no tiene pago (NaN)")

print("\n" + "-"*70)
print("\n3️⃣  RIGHT JOIN (todos los pagos)")
right = pd.merge(facturas, pagos, on='Factura', how='right')
print(right)
print(f"\nResultado: {len(right)} filas (TODOS los pagos, con/sin factura)")
print("\u26a0️  FAC-004 no tiene factura (pago sin factura)")

print("\n" + "-"*70)
print("\n4️⃣  OUTER JOIN (todo)")
outer = pd.merge(facturas, pagos, on='Factura', how='outer')
print(outer)
print(f"\nResultado: {len(outer)} filas (TODO - facturas Y pagos)")
print("\u26a0️  Muestra FAC-002 (sin pago) Y FAC-004 (sin factura)")

print("\n" + "="*70)
print("✅ Los 4 tipos de join dominados")

In [0]:
import pandas as pd

print("🔑 MERGE CON MÚLTIPLES CLAVES")
print("="*70)

# Ventas por Sucursal y Producto
ventas = pd.DataFrame({
    'Sucursal': ['Norte', 'Norte', 'Sur', 'Sur'],
    'Producto': ['Laptop', 'Mouse', 'Laptop', 'Teclado'],
    'Cantidad': [5, 50, 3, 30]
})

# Precios por Sucursal y Producto
precios = pd.DataFrame({
    'Sucursal': ['Norte', 'Norte', 'Sur', 'Sur'],
    'Producto': ['Laptop', 'Mouse', 'Laptop', 'Monitor'],
    'Precio': [80000, 1500, 75000, 12000]
})

print("\n📄 Ventas:")
print(ventas)
print("\n📄 Precios:")
print(precios)

print("\n" + "="*70)
print("\n1️⃣  MERGE CON 1 CLAVE (INCORRECTO)")
print("\n⚠️  Si hacemos merge solo por 'Producto':")
mal = pd.merge(ventas, precios, on='Producto', how='inner')
print(mal)
print("\n❌ Problema: Laptop Norte se combina con precio de Laptop Sur")

print("\n" + "="*70)
print("\n2️⃣  MERGE CON MÚLTIPLES CLAVES (CORRECTO)")
print("\n✅ Merge por Sucursal Y Producto:")
bien = pd.merge(ventas, precios, on=['Sucursal', 'Producto'], how='inner')
print(bien)

print("\n" + "-"*70)
print("\n3️⃣  CALCULAR TOTAL")
bien['Total'] = bien['Cantidad'] * bien['Precio']
print("\nVentas con Total calculado:")
print(bien)
print(f"\nTotal facturado: ${bien['Total'].sum():,.0f}")

print("\n" + "="*70)
print("✅ Merge con múltiples claves dominado")

In [0]:
import pandas as pd

print("✅ VALIDACIÓN DE MERGES")
print("="*70)

clientes = pd.DataFrame({
    'Cliente_ID': [1, 2, 3],
    'Nombre': ['Acme', 'TechStart', 'FinPlus']
})

pedidos = pd.DataFrame({
    'Cliente_ID': [1, 1, 2, 4],
    'Pedido': ['P001', 'P002', 'P003', 'P004'],
    'Monto': [50000, 30000, 25000, 15000]
})

print("\n📄 Clientes:")
print(clientes)
print("\n📄 Pedidos:")
print(pedidos)

print("\n" + "="*70)
print("\n1️⃣  INDICATOR (detectar origen de filas)")
merged = pd.merge(clientes, pedidos, on='Cliente_ID', how='outer', indicator=True)
print("\nMerge con indicator=True:")
print(merged)
print("\n📊 Resumen:")
print(merged['_merge'].value_counts())
print("\n⚠️  'right_only' = Pedido sin cliente (ID 4)")
print("    'left_only' = Cliente sin pedidos (ID 3)")

print("\n" + "="*70)
print("\n2️⃣  VALIDATE (verificar cardinalidad)")

print("\n✅ Validación 'one_to_many' (1 cliente → muchos pedidos):")
try:
    result = pd.merge(clientes, pedidos, on='Cliente_ID', how='inner', validate='one_to_many')
    print("  ✅ Validación exitosa")
    print(result)
except Exception as e:
    print(f"  ❌ Error: {e}")

print("\n" + "-"*70)
print("\n❌ Validación 'one_to_one' (debería fallar):")
try:
    result = pd.merge(clientes, pedidos, on='Cliente_ID', how='inner', validate='one_to_one')
    print("  ✅ Validación exitosa")
except Exception as e:
    print(f"  ❌ Error esperado: Hay cliente (ID 1) con 2 pedidos")

print("\n" + "="*70)
print("\n📊 OPCIONES DE VALIDATE")
print("-"*70)
print("  'one_to_one'   : 1 fila df1 → 1 fila df2")
print("  'one_to_many'  : 1 fila df1 → muchas filas df2")
print("  'many_to_one'  : muchas filas df1 → 1 fila df2")
print("  'many_to_many' : muchas filas df1 → muchas filas df2")
print("-"*70)

print("\n" + "="*70)
print("✅ Validación de merges dominada")

## 🏬 Conciliación Bancaria Automática

### 🎯 ¿Qué es una Conciliación Bancaria?

**Problema:**
* 📚 **Libro Diario:** Registros contables internos (facturas, pagos)
* 🏬 **Extracto Bancario:** Movimientos reales en el banco
* ❓ **Diferencias:** Cheques pendientes, errores, depósitos en tránsito

**Objetivo:** **Identificar automáticamente** qué está conciliado y qué no.

---

### 🛠️ Algoritmo de Conciliación

```python
1. Outer Merge (libro + extracto) por Nro_Comprobante
2. Calcular Diferencia = Monto_Libro - Monto_Banco
3. Clasificar:
   - Diferencia = 0     → "Conciliado"
   - Solo en Libro      → "Pendiente en Banco"
   - Solo en Extracto   → "Pendiente en Libro"
   - Diferencia ≠ 0     → "Diferencia de Monto"
```

---

### 📊 Casos Típicos

| Situación | Libro | Banco | Estado |
|-----------|-------|-------|--------|
| Factura pagada | $50K | $50K | ✅ Conciliado |
| Cheque no cobrado | $50K | - | ⚠️ Pendiente en Banco |
| Depósito no registrado | - | $50K | ⚠️ Pendiente en Libro |
| Error de monto | $50K | $49K | ❌ Diferencia ($1K) |

---

### 💡 Por Qué Outer Join

👉 **Outer join** nos permite ver:
* Lo que está en ambos (conciliado)
* Lo que solo está en el libro
* Lo que solo está en el extracto

**Inner join fallaría** porque solo mostraría lo conciliado.

In [0]:
import pandas as pd
import numpy as np

print("💼 CASO INTEGRADOR: CONCILIACIÓN BANCARIA AUTOMÁTICA")
print("="*70)

# Libro Diario (registros contables internos)
libro_diario = pd.DataFrame({
    'Fecha': pd.date_range('2024-01-01', periods=10),
    'Comprobante': ['FAC-001', 'FAC-002', 'FAC-003', 'CHQ-001', 'FAC-004',
                    'FAC-005', 'CHQ-002', 'FAC-006', 'FAC-007', 'DEP-001'],
    'Concepto': ['Venta Acme', 'Venta TechStart', 'Venta FinPlus', 'Pago Proveedor A',
                 'Venta DataCo', 'Venta LogiExp', 'Pago Proveedor B', 'Venta CloudSys',
                 'Venta NetCorp', 'Depósito efectivo'],
    'Monto_Libro': [150000, 85000, 42000, -30000, 190000,
                    75000, -45000, 62000, 88000, 50000]
})

# Extracto Bancario (movimientos reales)
extracto_bancario = pd.DataFrame({
    'Fecha_Banco': pd.date_range('2024-01-02', periods=9),
    'Comprobante': ['FAC-001', 'FAC-002', 'FAC-004', 'FAC-005', 'CHQ-002',
                    'FAC-006', 'FAC-007', 'FAC-099', 'DEP-001'],
    'Descripcion': ['Depósito Acme', 'Depósito TechStart', 'Depósito DataCo',
                    'Depósito LogiExp', 'Cheque Prov B', 'Depósito CloudSys',
                    'Depósito NetCorp', 'Depósito Desconocido', 'Efectivo'],
    'Monto_Banco': [150000, 85000, 190000, 75000, -45000,
                    62000, 87000, 25000, 50000]
})

print("\n📚 LIBRO DIARIO (10 movimientos):")
print(libro_diario)

print("\n🏬 EXTRACTO BANCARIO (9 movimientos):")
print(extracto_bancario)

print("\n" + "="*70)
print("\n🛠️ PROCESO DE CONCILIACIÓN")
print("-"*70)

print("\nPaso 1: Outer Merge por Comprobante")
conciliacion = pd.merge(
    libro_diario[['Comprobante', 'Concepto', 'Monto_Libro']],
    extracto_bancario[['Comprobante', 'Descripcion', 'Monto_Banco']],
    on='Comprobante',
    how='outer',
    indicator=True
)

print("\nPaso 2: Calcular Diferencia")
conciliacion['Diferencia'] = conciliacion['Monto_Libro'].fillna(0) - conciliacion['Monto_Banco'].fillna(0)

print("\nPaso 3: Clasificar Estado")
def clasificar_estado(row):
    if row['_merge'] == 'both':
        if row['Diferencia'] == 0:
            return '✅ Conciliado'
        else:
            return '❌ Diferencia de Monto'
    elif row['_merge'] == 'left_only':
        return '⚠️ Pendiente en Banco'
    else:  # right_only
        return '⚠️ Pendiente en Libro'

conciliacion['Estado'] = conciliacion.apply(clasificar_estado, axis=1)

print("\n" + "="*70)
print("\n✅ RESULTADO DE LA CONCILIACIÓN")
print("-"*70)
print(conciliacion[['Comprobante', 'Monto_Libro', 'Monto_Banco', 'Diferencia', 'Estado']])

print("\n" + "="*70)
print("\n📊 RESUMEN")
print("-"*70)
print(conciliacion['Estado'].value_counts())

print("\n" + "-"*70)
print("\n👉 ANÁLISIS:")
conciliados = conciliacion[conciliacion['Estado'] == '✅ Conciliado']
pendiente_banco = conciliacion[conciliacion['Estado'] == '⚠️ Pendiente en Banco']
pendiente_libro = conciliacion[conciliacion['Estado'] == '⚠️ Pendiente en Libro']
diferencias = conciliacion[conciliacion['Estado'] == '❌ Diferencia de Monto']

print(f"\n✅ Conciliados: {len(conciliados)} ({len(conciliados)/len(conciliacion)*100:.1f}%)")
print(f"   Total: ${conciliados['Monto_Libro'].sum():,.0f}")

if len(pendiente_banco) > 0:
    print(f"\n⚠️ Pendientes en Banco: {len(pendiente_banco)}")
    print("   Comprobantes:", pendiente_banco['Comprobante'].tolist())
    print("   Razón: Cheques no cobrados, depósitos en tránsito")

if len(pendiente_libro) > 0:
    print(f"\n⚠️ Pendientes en Libro: {len(pendiente_libro)}")
    print("   Comprobantes:", pendiente_libro['Comprobante'].tolist())
    print("   Razón: Movimientos bancarios no registrados")

if len(diferencias) > 0:
    print(f"\n❌ Diferencias de Monto: {len(diferencias)}")
    print(diferencias[['Comprobante', 'Monto_Libro', 'Monto_Banco', 'Diferencia']])

print("\n" + "="*70)
print("✅ Conciliación bancaria automática completada")

## 🎓 Conclusiones del Notebook 05_01

### ✅ Lo Que Aprendiste

1. **Los 4 Tipos de Join:**
   - `inner`: Solo coincidencias
   - `left`: Todas las filas de df1 + coincidencias
   - `right`: Todas las filas de df2 + coincidencias
   - `outer`: TODAS las filas de ambos

2. **Merge con Múltiples Claves:**
   ```python
   pd.merge(df1, df2, on=['col1', 'col2'], how='inner')
   ```

3. **Validación de Merges:**
   - `indicator=True`: Ver origen de filas (_merge)
   - `validate='one_to_many'`: Verificar cardinalidad

4. **Conciliación Bancaria:**
   - Outer merge por comprobante
   - Calcular diferencias
   - Clasificar estados automáticamente

---

### 🎯 Guía de Decisión de Join

| Pregunta | Join |
|----------|------|
| ¿Solo quiero coincidencias? | **inner** |
| ¿Quiero todo del lado izquierdo? | **left** |
| ¿Quiero todo del lado derecho? | **right** |
| ¿**Quiero ver diferencias**? | **outer** |

---

### 📊 Pipeline de Conciliación

```python
# 1. Outer merge
conc = pd.merge(libro, banco, on='Comprobante', how='outer', indicator=True)

# 2. Calcular diferencia
conc['Diff'] = conc['Monto_Libro'].fillna(0) - conc['Monto_Banco'].fillna(0)

# 3. Clasificar
conc['Estado'] = conc.apply(clasificar, axis=1)

# 4. Reportar
print(conc['Estado'].value_counts())
```

---

### 🚀 Próximo Notebook

**05_02 - Conciliación Automática Avanzada**
* Tolerancia de diferencias
* Coincidencia por fecha y monto
* Casos con múltiples criterios
* Automatización completa

---

<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🔗 ¡Merge y Conciliación Dominados!</h3>
  <p><i>"Outer merge + lógica de negocio = Conciliación automática en segundos."</i></p>
</div>